# Training Loop


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Users/montekkundan/Developer/ML/llm')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import torch
from course_tools import CharTokenizer, TinyConfig, TinyTransformerLM, train_model, evaluate_model

LECTURE_NOTE_PATH = Path('/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Training Loop.md')
print(LECTURE_NOTE_PATH)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Training Loop.md


## Demo A: the minimal loop is forward, backward, step, repeat


In [2]:
text = 'training loops turn data and gradients into learned weights.\n' * 8
tokenizer = CharTokenizer.build([text])
model = TinyTransformerLM(TinyConfig(vocab_size=len(tokenizer.stoi), block_size=32, d_model=32, n_heads=4, n_layers=1, d_ff=64))
history = train_model(model, tokenizer, train_text=text, eval_text=text, steps=10, learning_rate=3e-3, batch_size=4)
print(history)


[{'step': 1.0, 'train_loss': 21.06629180908203, 'eval_loss': 20.105648040771484, 'bpb': 29.006318722279925}, {'step': 2.0, 'train_loss': 20.69479751586914, 'eval_loss': 20.11833953857422, 'bpb': 29.024628683221383}, {'step': 4.0, 'train_loss': 19.175769805908203, 'eval_loss': 18.8814697265625, 'bpb': 27.24020273920681}, {'step': 6.0, 'train_loss': 17.078216552734375, 'eval_loss': 16.99983024597168, 'bpb': 24.52557079181755}, {'step': 8.0, 'train_loss': 17.15452766418457, 'eval_loss': 16.50352668762207, 'bpb': 23.809556109411023}, {'step': 10.0, 'train_loss': 15.447307586669922, 'eval_loss': 14.357209205627441, 'bpb': 20.713074521964085}]


## Demo B: validation belongs inside the run, not only at the end


In [3]:
metrics = evaluate_model(model, tokenizer, text)
print(metrics)


{'loss': 14.411062240600586, 'bpb': 20.79076802845666}


## Demo C: batch size and accumulation change effective tokens per update


In [4]:
seq_len = model.config.block_size
for micro_batch, accumulation in [(4, 1), (4, 4), (8, 2)]:
    tokens_per_update = micro_batch * accumulation * seq_len
    print({'micro_batch': micro_batch, 'accumulation': accumulation, 'tokens_per_update': tokens_per_update})


{'micro_batch': 4, 'accumulation': 1, 'tokens_per_update': 128}
{'micro_batch': 4, 'accumulation': 4, 'tokens_per_update': 512}
{'micro_batch': 8, 'accumulation': 2, 'tokens_per_update': 512}


## Demo D: checkpoints and reports are part of the loop


The runnable script for this lecture writes a checkpoint and a report under artifacts/training_loop_demo/.


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_train.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/base_train.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/checkpoint_manager.py
